In [ ]:
!pip install seaborn
!pip install graphviz
!pip install statsmodels
!pip install xgboost
!pip install lightgbm
!pip install catboost
!pip install mlxtend

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import graphviz

import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.api import qqplot, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, precision_score, r2_score, recall_score, roc_auc_score, silhouette_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

print("모든 패키지 import 성공")

In [ ]:
df_raw1 = pd.read_csv(r"../data/processed/bat_process.csv", encoding = "euc-kr")
display(df_raw1.head())
df_raw2 = pd.read_csv(r"../data/processed/bat_tat.csv", encoding = "euc-kr")
display(df_raw2.head())

In [ ]:
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# =========================================================
# 0. 한글 폰트 자동 설정
# =========================================================
def set_korean_font():
    preferred_fonts = [
        "Malgun Gothic",     # Windows
        "AppleGothic",       # macOS
        "NanumGothic",       # Linux common
        "NanumBarunGothic",
        "Noto Sans CJK KR",
        "Noto Sans KR"
    ]

    available_fonts = {f.name for f in fm.fontManager.ttflist}

    selected_font = None
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            selected_font = font_name
            break

    if selected_font is not None:
        plt.rcParams["font.family"] = selected_font
    else:
        # fallback: 기본 sans-serif
        plt.rcParams["font.family"] = "sans-serif"

    plt.rcParams["axes.unicode_minus"] = False
    return selected_font

selected_font = set_korean_font()

# =========================================================
# 1. 실행 폴더 생성
# =========================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = f"outlier_full_run_final_{timestamp}"
DATA_DIR = os.path.join(BASE_DIR, "data")
TABLE_DIR = os.path.join(BASE_DIR, "tables")
PLOT_DIR = os.path.join(BASE_DIR, "plots")
LOG_DIR = os.path.join(BASE_DIR, "logs")

for d in [BASE_DIR, DATA_DIR, TABLE_DIR, PLOT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# =========================================================
# 2. 입력 파일 자동 탐색
# =========================================================
def find_latest_file(filename_pattern: str) -> str:
    matches = glob.glob(f"**/{filename_pattern}", recursive=True)
    matches = [m for m in matches if os.path.isfile(m)]
    if len(matches) == 0:
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {filename_pattern}")
    matches = sorted(matches, key=lambda x: os.path.getmtime(x), reverse=True)
    return matches[0]

RAW_FILE = find_latest_file("bat_process.csv")

used_files = pd.DataFrame({
    "role": ["RAW_FILE"],
    "path": [RAW_FILE]
})
used_files.to_csv(os.path.join(LOG_DIR, "used_files.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 3. 데이터 로드
# =========================================================
df = pd.read_csv(RAW_FILE, encoding="euc-kr")

if "judge" not in df.columns:
    raise ValueError("judge 컬럼이 필요합니다.")

# 원본 구조 유지용 최종 데이터
df_processed = df.copy()

# 분석/flag/시각화용 별도 데이터프레임
df_work = df.copy()
df_work["judge_bin"] = (df_work["judge"] == "불량").astype(int)

n_rows = len(df_work)
baseline_fail_rate = df_work["judge_bin"].mean()

# =========================================================
# 4. 변수 그룹 정의
# =========================================================
A_DATA_ERROR = [
    "ocv1_ocv", "ocv2_ocv", "socv1_ocv", "socv2_ocv", "socv3_ocv",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "pg1_imp", "pg1_impfit", "pc1_imp", "m1_res_ac",
    "m1_thick",
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
]

B_RECIPE = [
    "c1_curr_end", "dc1_curr_end", "c2_curr_end", "dc2_curr_end",
    "c3_curr_end", "dc3_curr_end", "c4_curr_end",
    "c3_cvval", "c4_cvval",
    "c3_ccval", "c4_ccval",
]

C_PROCESS = [
    "ocv2_deltaocv", "pg1_imp", "pc1_imp", "m1_res_ac", "m1_thick",
    "c3_time_cv", "c4_time_cv",
]

D_NORMAL = [
    "c1_temp_avg", "dc1_temp_avg", "c2_temp_avg", "dc2_temp_avg",
    "c3_temp_avg", "dc3_temp_avg", "c4_temp_avg",
    "c1_time_cc", "c2_time_cc", "c3_time_cc", "c4_time_cc",
    "c3_time_cv", "c4_time_cv",
    "c1_voltage_avg", "dc1_voltage_avg",
    "c2_voltage_avg", "dc2_voltage_avg",
    "c3_voltage_avg", "dc3_voltage_avg",
    "c4_voltage_avg",
]

A_DATA_ERROR = [c for c in A_DATA_ERROR if c in df_work.columns]
B_RECIPE = [c for c in B_RECIPE if c in df_work.columns]
C_PROCESS = [c for c in C_PROCESS if c in df_work.columns]
D_NORMAL = [c for c in D_NORMAL if c in df_work.columns]

# =========================================================
# 5. B형 설정
# =========================================================
B_CONFIG = {
    "c3_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},
    "c4_cvval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": 20, "tol_ratio": None},

    "c1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc1_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc2_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "dc3_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},
    "c4_curr_end": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.02},

    "c3_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
    "c4_ccval": {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03},
}

# =========================================================
# 6. 유틸 함수
# =========================================================
def apply_hard_limit(series, low=None, high=None):
    s = series.copy()
    if low is not None:
        s = s.mask(s < low, np.nan)
    if high is not None:
        s = s.mask(s > high, np.nan)
    return s

def iqr_bounds(series, k=1.5):
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return low, high

def iqr_clip(series, k=1.5):
    low, high = iqr_bounds(series, k=k)
    return series.clip(lower=low, upper=high), low, high

def get_main_recipe_centers(series, top_k=3, min_ratio=0.05, round_digits=None):
    s = series.dropna().copy()
    if len(s) == 0:
        return []
    if round_digits is not None:
        s = s.round(round_digits)
    vc = s.value_counts(normalize=True)
    centers = vc[vc >= min_ratio].index.tolist()[:top_k]
    if len(centers) == 0:
        centers = vc.index.tolist()[:1]
    return sorted(centers)

def make_recipe_flag(series, centers, tol_abs=None, tol_ratio=None):
    s = series.copy()
    if len(centers) == 0:
        return pd.Series(0, index=s.index, dtype=int)

    ok_mask = pd.Series(False, index=s.index)
    for c in centers:
        tol = tol_abs if tol_abs is not None else abs(c) * tol_ratio
        ok_mask = ok_mask | ((s >= c - tol) & (s <= c + tol))
    return (~ok_mask).astype(int)

# =========================================================
# 7. 처리 및 요약
# =========================================================
summary_rows = []
flag_store = {}

# -------------------------
# A형: sanity check
# -------------------------
A_LIMITS = {}
for col in A_DATA_ERROR:
    if "temp" in col:
        A_LIMITS[col] = (0, 1000)
    elif "time" in col:
        A_LIMITS[col] = (0, 100000)
    elif "ocv" in col or "voltage" in col:
        A_LIMITS[col] = (0, 5000)
    elif "imp" in col or "res" in col:
        A_LIMITS[col] = (0, 10000)
    elif "thick" in col:
        A_LIMITS[col] = (0, 10000)
    else:
        A_LIMITS[col] = (None, None)

for col in A_DATA_ERROR:
    before = df_processed[col].copy()
    low, high = A_LIMITS[col]
    after = apply_hard_limit(before, low=low, high=high)

    df_processed[col] = after
    df_work[col] = after

    changed = before.notna().sum() - after.notna().sum()

    summary_rows.append({
        "column": col,
        "type": "A_data_error",
        "action": "hard_limit_to_nan",
        "low": low,
        "high": high,
        "changed_count": int(changed),
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

# -------------------------
# B형: recipe deviation
# -------------------------
for col in B_RECIPE:
    s = df_work[col].dropna()
    if len(s) == 0:
        continue

    cfg = B_CONFIG.get(col, {"round_digits": 0, "top_k": 3, "min_ratio": 0.03, "tol_abs": None, "tol_ratio": 0.03})
    centers = get_main_recipe_centers(
        s,
        top_k=cfg["top_k"],
        min_ratio=cfg["min_ratio"],
        round_digits=cfg["round_digits"]
    )

    flag = make_recipe_flag(
        series=df_work[col],
        centers=centers,
        tol_abs=cfg["tol_abs"],
        tol_ratio=cfg["tol_ratio"]
    )

    flag_name = f"{col}_flag_recipe"
    flag_store[flag_name] = flag

    changed = int(flag.sum())

    if cfg["tol_abs"] is not None:
        low_display = min(centers) - cfg["tol_abs"] if len(centers) else np.nan
        high_display = max(centers) + cfg["tol_abs"] if len(centers) else np.nan
    else:
        low_display = min(centers) * (1 - cfg["tol_ratio"]) if len(centers) else np.nan
        high_display = max(centers) * (1 + cfg["tol_ratio"]) if len(centers) else np.nan

    summary_rows.append({
        "column": col,
        "type": "B_recipe_deviation",
        "action": "flag_only_mode_centers",
        "low": low_display,
        "high": high_display,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ",".join(map(str, centers))
    })

# -------------------------
# C형: process anomaly
# -------------------------
for col in C_PROCESS:
    s = df_work[col].dropna()
    if len(s) == 0:
        continue

    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    flag95 = (df_work[col] > p95).astype(int)
    flag99 = (df_work[col] > p99).astype(int)

    flag_store[f"{col}_flag_p95"] = flag95
    flag_store[f"{col}_flag_p99"] = flag99

    changed = int(flag99.sum())

    summary_rows.append({
        "column": col,
        "type": "C_process_anomaly",
        "action": "flag_only_p95_p99",
        "low": p95,
        "high": p99,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

# -------------------------
# D형: clip + flag
# -------------------------
clip_compare_rows = []

for col in D_NORMAL:
    before = df_processed[col].copy()
    clipped, low, high = iqr_clip(before, k=1.5)

    df_processed[col] = clipped
    df_work[col] = clipped

    flag_iqr = ((before < low) | (before > high)).astype(int)
    flag_store[f"{col}_flag_iqr"] = flag_iqr

    changed = int(flag_iqr.sum())

    summary_rows.append({
        "column": col,
        "type": "D_normal_variation",
        "action": "iqr_clip",
        "low": low,
        "high": high,
        "changed_count": changed,
        "changed_ratio": changed / n_rows,
        "recipe_centers": ""
    })

    clip_compare_rows.append({
        "column": col,
        "before_q95": before.quantile(0.95),
        "after_q95": clipped.quantile(0.95),
        "before_q99": before.quantile(0.99),
        "after_q99": clipped.quantile(0.99)
    })

# =========================================================
# 8. 요약표 / flag표 저장
# =========================================================
summary = pd.DataFrame(summary_rows).sort_values(["type", "changed_ratio"], ascending=[True, False])
summary.to_csv(os.path.join(TABLE_DIR, "outlier_final_summary.csv"), index=False, encoding="utf-8-sig")

type_summary = (
    summary.groupby(["type", "action"], as_index=False)
    .agg(
        variable_count=("column", "count"),
        total_changed=("changed_count", "sum"),
        mean_changed_ratio=("changed_ratio", "mean"),
        max_changed_ratio=("changed_ratio", "max")
    )
)
type_summary["mean_changed_ratio_pct"] = type_summary["mean_changed_ratio"] * 100
type_summary["max_changed_ratio_pct"] = type_summary["max_changed_ratio"] * 100
type_summary.to_csv(os.path.join(TABLE_DIR, "type_summary.csv"), index=False, encoding="utf-8-sig")

flag_df = pd.DataFrame(flag_store)
flag_df.to_csv(os.path.join(TABLE_DIR, "outlier_flags_only.csv"), index=False, encoding="utf-8-sig")

flag_summary = pd.DataFrame({
    "flag_col": flag_df.columns,
    "flag_count": [flag_df[c].sum() for c in flag_df.columns],
    "flag_ratio": [flag_df[c].mean() for c in flag_df.columns]
}).sort_values("flag_ratio", ascending=False)
flag_summary["flag_ratio_pct"] = flag_summary["flag_ratio"] * 100
flag_summary.to_csv(os.path.join(TABLE_DIR, "flag_summary.csv"), index=False, encoding="utf-8-sig")

pd.DataFrame(clip_compare_rows).to_csv(
    os.path.join(TABLE_DIR, "clip_quantile_compare.csv"),
    index=False, encoding="utf-8-sig"
)

# 원본 구조 유지한 최종 결과
df_processed.to_csv(os.path.join(DATA_DIR, "bat_process_outlier_final.csv"), index=False, encoding="utf-8-sig")

# =========================================================
# 9. 대표 시각화 1: B형 히스토그램 + recipe centers
# =========================================================
b_cols = ["c3_cvval", "c4_cvval", "c3_ccval", "c1_curr_end"]
b_cols = [c for c in b_cols if c in df_processed.columns]

fig, axes = plt.subplots(len(b_cols), 1, figsize=(9, 4 * max(1, len(b_cols))))
if len(b_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, b_cols):
    s = df_processed[col].dropna()
    ax.hist(s, bins=40, alpha=0.8)

    row = summary[summary["column"] == col]
    centers = []
    changed_ratio = np.nan

    if len(row) > 0:
        changed_ratio = row.iloc[0]["changed_ratio"] * 100
        centers_text = str(row.iloc[0].get("recipe_centers", ""))
        if centers_text and centers_text != "nan":
            for x in centers_text.split(","):
                try:
                    centers.append(float(x))
                except:
                    pass

    for c in centers:
        ax.axvline(c, linestyle="--", linewidth=1.5)

    flag_col = f"{col}_flag_recipe"
    flag_ratio = flag_df[flag_col].mean() * 100 if flag_col in flag_df.columns else np.nan

    ax.set_title(f"{col} | flagged={flag_ratio:.2f}% | changed={changed_ratio:.2f}%")
    ax.set_xlabel(col)
    ax.set_ylabel("개수")

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_B_recipe_hist.png"))
plt.close()

# =========================================================
# 10. 대표 시각화 2: C형 히스토그램 + 양품/불량 boxplot
# =========================================================
c_cols = ["pg1_imp", "c4_time_cv"]
c_cols = [c for c in c_cols if c in df_processed.columns]

fig, axes = plt.subplots(len(c_cols), 2, figsize=(14, 5 * max(1, len(c_cols))))
if len(c_cols) == 1:
    axes = np.array([axes])

for i, col in enumerate(c_cols):
    s = df_processed[col].dropna()
    p95 = s.quantile(0.95)
    p99 = s.quantile(0.99)

    axes[i, 0].hist(s, bins=40, alpha=0.8)
    axes[i, 0].axvline(p95, linestyle="--", linewidth=1.5, label=f"p95={p95:.2f}")
    axes[i, 0].axvline(p99, linestyle="--", linewidth=1.5, label=f"p99={p99:.2f}")
    axes[i, 0].set_title(f"{col} - 히스토그램과 실제 p95/p99")
    axes[i, 0].legend()

    good = df_processed.loc[df_processed["judge"] != "불량", col].dropna()
    bad = df_processed.loc[df_processed["judge"] == "불량", col].dropna()
    axes[i, 1].boxplot([good, bad], labels=["양품", "불량"])
    axes[i, 1].set_title(f"{col} - 양품/불량 박스플롯")

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_C_hist_box.png"))
plt.close()

# =========================================================
# 11. 대표 시각화 3: D형 before/after 히스토그램 + 박스플롯
# =========================================================
d_cols = ["c3_time_cc", "c2_voltage_avg"]
d_cols = [c for c in d_cols if c in df.columns and c in df_processed.columns]

fig, axes = plt.subplots(len(d_cols), 2, figsize=(14, 5 * max(1, len(d_cols))))
if len(d_cols) == 1:
    axes = np.array([axes])

for i, col in enumerate(d_cols):
    before = df[col].dropna()
    after = df_processed[col].dropna()

    axes[i, 0].hist(before, bins=40, alpha=0.6, label="Before")
    axes[i, 0].hist(after, bins=40, alpha=0.6, label="After")
    axes[i, 0].set_title(f"{col} - 처리 전/후 히스토그램")
    axes[i, 0].legend()

    axes[i, 1].boxplot([before, after], labels=["Before", "After"])
    axes[i, 1].set_title(f"{col} - 처리 전/후 박스플롯")

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_D_hist_box_before_after.png"))
plt.close()

# =========================================================
# 12. 대표 시각화 4: flag별 불량률 uplift
# =========================================================
flag_effect_rows = []
for col in flag_df.columns:
    if flag_df[col].nunique() < 2:
        continue

    tmp = pd.DataFrame({
        "flag": flag_df[col],
        "judge_bin": df_work["judge_bin"]
    })

    fail0 = tmp.loc[tmp["flag"] == 0, "judge_bin"].mean()
    fail1 = tmp.loc[tmp["flag"] == 1, "judge_bin"].mean()

    flag_effect_rows.append({
        "flag_col": col,
        "fail_rate_flag0": fail0,
        "fail_rate_flag1": fail1,
        "uplift": fail1 - fail0,
        "prevalence_pct": tmp["flag"].mean() * 100
    })

flag_effect = pd.DataFrame(flag_effect_rows).sort_values("uplift", ascending=False)
flag_effect.to_csv(os.path.join(TABLE_DIR, "flag_failrate_uplift.csv"), index=False, encoding="utf-8-sig")

top_flag_effect = flag_effect.head(12)

plt.figure(figsize=(10, 6))
plt.barh(top_flag_effect["flag_col"], top_flag_effect["uplift"])
plt.gca().invert_yaxis()
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("불량률 상승폭 (flag=1 - flag=0)")
plt.title("flag별 불량률 상승 효과")
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_flag_failrate_uplift.png"))
plt.close()

# =========================================================
# 13. 로그 저장
# =========================================================
meta_info = pd.DataFrame({
    "item": ["selected_font", "raw_rows", "raw_cols", "processed_rows", "processed_cols", "baseline_fail_rate"],
    "value": [selected_font, df.shape[0], df.shape[1], df_processed.shape[0], df_processed.shape[1], baseline_fail_rate]
})
meta_info.to_csv(os.path.join(LOG_DIR, "meta_info.csv"), index=False, encoding="utf-8-sig")

notes = [
    "최종 처리 데이터는 원본 컬럼 구조를 그대로 유지하고, 실제 값만 처리하였다.",
    "flag 컬럼들은 별도 파일(outlier_flags_only.csv)로 분리 저장하였다.",
    "A형은 데이터 파손 여부 확인용 sanity check다.",
    "B형은 recipe center 기반 이탈 탐지다.",
    "C형은 p95/p99 tail 기반 위험 신호 보존이다.",
    "D형은 IQR 기반 완만한 clip이다.",
    "그래프 한글 깨짐을 막기 위해 실행 환경에서 사용 가능한 한글 폰트를 자동 설정했다."
]
with open(os.path.join(LOG_DIR, "final_notes.txt"), "w", encoding="utf-8") as f:
    for line in notes:
        f.write(line + "\n")

# =========================================================
# 14. 완료 메시지
# =========================================================
print("\n완료 ✅")
print(f"결과 폴더: {BASE_DIR}")
print(f"사용 한글 폰트: {selected_font}")
print(f"\n데이터 폴더: {DATA_DIR}")
print(f"표 폴더: {TABLE_DIR}")
print(f"그래프 폴더: {PLOT_DIR}")
print(f"로그 폴더: {LOG_DIR}")

print("\n핵심 결과 파일")
print("- data/bat_process_outlier_final.csv  ← 원본 구조 유지, 값만 처리")
print("- tables/outlier_flags_only.csv       ← flag 컬럼 별도 저장")
print("- tables/outlier_final_summary.csv")
print("- tables/type_summary.csv")
print("- tables/flag_summary.csv")
print("- plots/01_B_recipe_hist.png")
print("- plots/02_C_hist_box.png")
print("- plots/03_D_hist_box_before_after.png")
print("- plots/04_flag_failrate_uplift.png")

In [ ]:
total = len(df)
fail = (df['judge'] == '불량').sum()

fail_rate = fail / total

print(f"전체: {total}")
print(f"불량: {fail}")
print(f"불량률: {fail_rate:.4f} ({fail_rate*100:.2f}%)")